In [ ]:
# Import packages, load the study configuration and the annotated sample table, and
# set the input and output locations
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.burden import analysis as mut_burden_funcs
from SSc_Somatic.burden import plots as mut_burden_plots_funcs

%matplotlib inline

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'mutation')

# Annotated sample table (SNV counts merged with metadata) and case subtype IDs
meta_df = data.samples
subtype_a, subtype_b = config.require('cohort', 'subtypes')
limited_ids = data.group_ids[subtype_a]
diffuse_ids = data.group_ids[subtype_b]

# Input and output locations
tidy_vcf_filename = config.output_path('mutation', 'tidy_variants')
plot_root = str(config.output_dir('mutation', 'figures_dir')) + os.sep
gwas_output_root = str(config.output_dir('mutation', 'gwas_validation_dir'))

In [ ]:
# Load the tidy variant table written by 02_build_variant_table
variants_df = pd.read_csv(tidy_vcf_filename).rename(columns={"SAMPLE": "sample_id"})

# Sample IDs stored both as numbers and as text are duplicates left over from a
# merge; they are removed from every count table below
numeric_samples = set(str(int(x)) for x in variants_df['sample_id'].unique() if isinstance(x, (int, float)))
string_samples = set(x for x in variants_df['sample_id'].unique() if isinstance(x, str))
dupe_ids = list(numeric_samples & string_samples)
print(f"Number of sample IDs stored both as numbers and as text (removed below): {len(dupe_ids)}")


def burden_table(variants, count_col):
    """Count variants per sample, drop duplicated sample IDs and build the analysis table."""
    counts = variants.groupby('sample_id').size().reset_index(name=count_col)
    counts = counts[~counts['sample_id'].isin(dupe_ids)]
    print(f"Samples after removing duplicated IDs: {len(counts)}")
    table = mut_burden_funcs.create_analysis_df(meta_df, counts, mutation_source='vcf', burden_col=count_col)
    return mut_burden_funcs.prepare_analysis_df(table, burden_col='mutation_burden')


# SNVs only (single base changes)
snv_mask = (
    (variants_df['REF'].str.len() == 1) &
    (variants_df['ALT'].str.len() == 1)
)
variants_snv = variants_df[snv_mask].copy()
print("Building the SNV burden table for all SNVs:")
analysis_df = burden_table(variants_snv, 'snv_count_vcf')

# Copy for plotting (the analysis table is already prepared)
plot_df = analysis_df.copy()

print('Discrepancies in metadata/mutation data sample numbers due to filtering (PON, outlier, multiple ana etc.)')

All SNVs, including synonymous variants.

In [ ]:
# Plot SNV burden by disease group, for all SNVs
print("Figure: SNV burden by disease group (all SNVs)")
mut_burden_plots_funcs.plot_mutation_burden(analysis_df, save_path=plot_root + 'SSc_fig1B.pdf')

In [ ]:
# Compute stats
disease_stats = mut_burden_funcs.compute_burden_age_correlation_by_disease(plot_df)

# Plot SNV count by age of sequencing for each subtype
print("Figure: SNV burden against age at sequencing in each disease group, with Spearman "
      "correlations (all SNVs)")
fig_a, axes_a = mut_burden_plots_funcs.plot_burden_vs_age_by_disease(
    plot_df,
    stats_df=disease_stats,
    output_filename=None
)

In [ ]:
# Full figure; legend placement was adjusted manually after export
print("Figure: age-of-onset distributions, SNV burden against disease duration, and SNV burden "
      "against age compared with the control regression for disease durations of 2 years or "
      "less, 3 to 5 years, 6 to 10 years and all durations (all SNVs)")
fig = mut_burden_plots_funcs.create_full_multipanel_figure(analysis_df, save_path=plot_root + 'SSc_sup_fig2.pdf')

In [ ]:
# Compute stats
race_stats = mut_burden_funcs.compute_burden_age_correlation_by_race_within_disease(plot_df)

# Plot SNV count by age of sequencing for each subtype for each race
print("Figure: SNV burden against age at sequencing by race within each disease group, with "
      "Spearman correlations (all SNVs)")
fig_b, axes_b = mut_burden_plots_funcs.plot_burden_vs_age_by_race_within_disease(
    plot_df,
    stats_df=race_stats,
    output_filename=None
)

In [ ]:
# Compute burden stats by onset, duration, and race
onset_duration_stats = mut_burden_funcs.compute_burden_stats_by_onset_duration_race(
    plot_df,
    n_bootstrap=1000
)

# Create summary table
summary_table = mut_burden_funcs.create_burden_summary_table(onset_duration_stats)

# Plot duration-stratified figure
print("Figure: mean SNV burden with 95% confidence intervals by age-of-onset group, disease "
      "duration and race (all SNVs)")
fig_dur, control_regs = mut_burden_plots_funcs.plot_burden_by_onset_duration_race(
    plot_df,
    onset_duration_stats,
    output_filename=plot_root + 'SSc_fig1C_fig1D.pdf'
)

In [ ]:
# Show the burden summary table by onset group, disease duration and race
print("Mean SNV burden with 95% confidence intervals by age-of-onset group, disease duration "
      "and race (all SNVs):")
print(summary_table)

In [ ]:
# Test whether cases have higher SNV burden than predicted by the age regression in
# controls, within each race group
print("Case SNV burden compared with the control age regression, by race (all SNVs):")
elevation_results, ctrl_regs = mut_burden_funcs.test_case_vs_control_elevation(analysis_df)

In [ ]:
# Test whether the case-control burden elevation is larger in young-onset than in
# older-onset cases, within each race group (baseline cases only)
print("Burden elevation in young-onset compared with older-onset cases, by race (all SNVs):")
younger_vs_older_results = mut_burden_funcs.test_young_vs_old_elevation(analysis_df)

In [ ]:
# Test whether disease duration is associated with SNV burden after adjusting for age,
# within each race group
print("Association of disease duration with SNV burden, adjusted for age, by race (all SNVs):")
duration_results, duration_model = mut_burden_funcs.test_duration_effect(analysis_df)

Potentially protein-altering SNVs only. Synonymous, unknown and unannotated ('.') variants are removed; note that '.' also covers splice-region variants.

In [ ]:
# Potentially protein-altering SNVs
before_syn = len(variants_snv)
variants_snv_prot = variants_snv[~variants_snv['FUNC'].isin(['synonymous_SNV','synonymous SNV','unknown','.'])]
print("Variant counts before and after removing synonymous, unknown and unannotated SNVs:")
print(f'Total variants: {len(variants_df):,}')
print(f'SNV variants: {before_syn:,}')
print(f'SNVs after synonymous removal: {len(variants_snv_prot):,}')

print("Building the SNV burden table for protein-altering SNVs:")
analysis_df_prot = burden_table(variants_snv_prot, 'snv_count_vcf')

# Copy for plotting (the analysis table is already prepared)
plot_df_prot = analysis_df_prot.copy()

# Look at snv breakdown
print("Number of protein-altering SNVs by exonic function:")
print(variants_snv_prot['FUNC'].value_counts())

In [ ]:
# Compute stats
disease_stats_prot = mut_burden_funcs.compute_burden_age_correlation_by_disease(plot_df_prot)

# Plot SNV count by age of sequencing for each subtype
print("Figure: SNV burden against age at sequencing in each disease group, with Spearman "
      "correlations (protein-altering SNVs)")
fig_a, axes_a = mut_burden_plots_funcs.plot_burden_vs_age_by_disease(
    plot_df_prot,
    stats_df=disease_stats_prot,
    output_filename=None
)

In [ ]:
# Compute stats
race_stats_prot = mut_burden_funcs.compute_burden_age_correlation_by_race_within_disease(plot_df_prot)

# Plot SNV count by age of sequencing for each subtype for each race
print("Figure: SNV burden against age at sequencing by race within each disease group, with "
      "Spearman correlations (protein-altering SNVs)")
fig_b, axes_b = mut_burden_plots_funcs.plot_burden_vs_age_by_race_within_disease(
    plot_df_prot,
    stats_df=race_stats_prot,
    output_filename=None
)

In [ ]:
# Compute burden stats by onset, duration, and race
onset_duration_stats_prot = mut_burden_funcs.compute_burden_stats_by_onset_duration_race(
    plot_df_prot,
    n_bootstrap=1000
)

# Create summary table
summary_table_prot = mut_burden_funcs.create_burden_summary_table(onset_duration_stats_prot)

# Plot duration-stratified figure
print("Figure: mean SNV burden with 95% confidence intervals by age-of-onset group, disease "
      "duration and race (protein-altering SNVs)")
mut_burden_plots_funcs.plot_burden_by_onset_duration_race(
    plot_df_prot,
    onset_duration_stats_prot,
    output_filename=plot_root + 'SNV_burden_over_duration_windows_prot.pdf',
    ylim=(25, 175)
)

In [ ]:
# Full figure; legend placement was adjusted manually after export
print("Figure: age-of-onset distributions, SNV burden against disease duration, and SNV burden "
      "against age compared with the control regression for disease durations of 2 years or "
      "less, 3 to 5 years, 6 to 10 years and all durations (protein-altering SNVs)")
mut_burden_plots_funcs.create_full_multipanel_figure(analysis_df_prot, save_path=plot_root + 'SSc_sup_fig3.pdf', ylim=(25, 175))

In [ ]:
# Test whether cases have higher protein-altering SNV burden than predicted by the age
# regression in controls, within each race group
print("Case SNV burden compared with the control age regression, by race "
      "(protein-altering SNVs):")
mut_burden_funcs.test_case_vs_control_elevation(analysis_df_prot)

In [ ]:
# Test whether the case-control burden elevation is larger in young-onset than in
# older-onset cases, within each race group (baseline cases only)
print("Burden elevation in young-onset compared with older-onset cases, by race "
      "(protein-altering SNVs):")
younger_vs_older_results = mut_burden_funcs.test_young_vs_old_elevation(analysis_df_prot)

In [ ]:
# Test whether disease duration is associated with protein-altering SNV burden after
# adjusting for age, within each race group
print("Association of disease duration with SNV burden, adjusted for age, by race "
      "(protein-altering SNVs):")
duration_results, duration_model = mut_burden_funcs.test_duration_effect(analysis_df_prot)

Now let's look at what these mutations are involved in

In [ ]:
# GO biological process enrichment of the genes carrying somatic SNVs, comparing cases
# with controls. gseapy downloads the gene-set library, so this needs internet access.
# Define case IDs for NK analysis
case_ids = list(set(limited_ids + diffuse_ids))
print(f"Case IDs: {len(case_ids)} ({subtype_a} + {subtype_b})")

# Gene coordinate file for length normalization
gff_path = config.path('reference', 'gene_annotation_gff')

# The enrichment functions expect the sample column to be named SAMPLE
analysis_df['SAMPLE'] = analysis_df['sample_id'].copy()
variants_snv['SAMPLE'] = variants_snv['sample_id'].copy()
analysis_df_prot['SAMPLE'] = analysis_df_prot['sample_id'].copy()
variants_snv_prot['SAMPLE'] = variants_snv_prot['sample_id'].copy()

# Run full GO:BP enrichment pipeline
# This performs multiple gseapy calls for all specified comparisons
print("Running GO biological process enrichment of the mutated genes:")
gobp_results = mut_burden_funcs.run_complete_gobp_analysis(
    variants_df=variants_snv,
    analysis_df=analysis_df,
    output_dir=str(config.output_dir('mutation', 'enrichment_dir')),
    gff_path=str(gff_path)
)

In [ ]:
# NK cell gene-set membership, variant counts per NK cell GO:BP term, and the cases that
# carry protein-altering variants in NK cell genes
print("Looking up the genes in each NK cell GO:BP term:")
nk_membership = mut_burden_funcs.get_nk_geneset_membership()
print("NK cell GO:BP term membership of each gene:")
print(nk_membership)

# Count protein-altering variants per NK GO:BP term
print("Counting low-VAF protein-altering variants in cases for each NK cell GO:BP term:")
nk_term_counts = mut_burden_funcs.count_variants_per_nk_goterm(
    tidy_vcf_df=variants_snv_prot,
    case_ids=case_ids,
    membership_df=nk_membership
)
print("Variant counts per NK cell GO:BP term:")
print(nk_term_counts)

# Classify NK mutation carriers
print("Identifying cases that carry low-VAF protein-altering variants in NK cell genes:")
nk_carrier_ids = mut_burden_funcs.classify_nk_mutation_carriers(
    tidy_vcf_df=variants_snv_prot,
    case_ids=case_ids
)

In [ ]:
# Clinical comparison: carriers vs non-carriers
print("Comparing clinical and demographic features between NK mutation carriers and non-carriers:")
nk_clinical = mut_burden_funcs.compare_nk_carriers_vs_noncarriers(
    meta_df=meta_df,
    carrier_ids=nk_carrier_ids,
    case_ids=case_ids
)
print("Clinical and demographic comparison of NK mutation carriers and non-carriers:")
print(nk_clinical)

In [ ]:
# Track specific driving variants
print("Listing the qualifying variants in NK cell genes:")
nk_variants = mut_burden_funcs.get_nk_variant_details(
    tidy_vcf_df=variants_snv_prot,
    case_ids=case_ids,
    membership_df=nk_membership
)
print("Qualifying NK cell gene variants, with their frequency and GO term membership:")
print(nk_variants)

In [ ]:
# Validate GWAS summary statistics
print("Loading and checking the GWAS summary statistics:")
gwas_df = mut_burden_funcs.validate_gwas_sumstats()

In [ ]:
# Prepare FUMA-formatted summary stats
print("Writing the GWAS summary statistics in FUMA format:")
fuma_path = mut_burden_funcs.prepare_fuma_sumstats(gwas_df=gwas_df)

In [ ]:
# Write a second FUMA file without the allele columns
fuma_fix = gwas_df[['SNP', 'CHR', 'BP', 'P', 'OR']].copy()
fuma_fix = fuma_fix.dropna(subset=['SNP', 'P'])
fuma_fix_path = os.path.join(gwas_output_root, 'fuma_ssc_sumstats_noalleles.txt')
fuma_fix.to_csv(fuma_fix_path, sep='\t', index=False)
print("FUMA summary statistics file without allele columns:")
print(f"Written: {fuma_fix_path}")
print(f"SNPs: {len(fuma_fix):,}")

In [ ]:
# Prepare MAGMA custom gene set file
print("Writing the custom gene-set file in MAGMA format:")
magma_path = mut_burden_funcs.prepare_magma_geneset_file()

Run MAGMA on the files written above (outside this notebook), then continue below.

In [ ]:
# Build Entrez ID list for the GO:BP term to map genes
import gseapy as gp

library = gp.get_library(config.require('enrichment', 'gseapy_library'))
go_term_key = [k for k in library.keys() if 'GO:0042267' in k][0]
go_symbols = library[go_term_key]
print("Genes in the NK cell mediated cytotoxicity term (GO:0042267) and their mapping to Entrez IDs:")
print(f"{go_term_key}: {len(go_symbols)} genes")

gene_loc_path = config.path('reference', 'magma_gene_locations')
loc = pd.read_csv(gene_loc_path, sep=r'\s+', header=None,
                   names=['ENTREZ', 'CHR', 'START', 'STOP', 'STRAND', 'SYMBOL'])
symbol_to_entrez = dict(zip(loc['SYMBOL'], loc['ENTREZ']))
entrez_to_symbol = dict(zip(loc['ENTREZ'], loc['SYMBOL']))

go_entrez_ids = [symbol_to_entrez[s] for s in go_symbols if s in symbol_to_entrez]
unmapped = [s for s in go_symbols if s not in symbol_to_entrez]
print(f"Mapped to Entrez: {len(go_entrez_ids)}, Unmapped: {len(unmapped)}")
if unmapped:
    print(f"  Unmapped symbols: {unmapped}")

In [ ]:
# Forest plot of MAGMA gene-set BETAs
gsa_hybrid_path = config.output_path('mutation', 'magma_gene_set_results', create_parent=False)

print("Figure: forest plot of MAGMA gene-set effect sizes (BETA) in the SSc GWAS")
fig_forest = mut_burden_plots_funcs.plot_magma_geneset_forest(
    gsa_path=str(gsa_hybrid_path),
    save_path=plot_root + 'magma_geneset_forest_hybrid.pdf'
)
plt.show()

Indel burden, for comparison.

In [ ]:
# Indels only (REF or ALT longer than 1 base)
indel_mask = (
    (variants_df['REF'].str.len() > 1) |
    (variants_df['ALT'].str.len() > 1)
)
variants_indel = variants_df[indel_mask].copy()
print("Building the indel burden table:")
analysis_df_indel = burden_table(variants_indel, 'indel_count_vcf')

print("Figure: indel burden by disease group")
fig, ax = mut_burden_plots_funcs.plot_mutation_burden(analysis_df_indel)

In [ ]:
# Cohort characteristics: age and age-of-onset distributions by race and disease group
print("Figure: distributions of age and age of onset by race and disease group")
mut_burden_plots_funcs.create_cohort_dist_plots(analysis_df, save_path=None)